# 🐝 Lab 2.3 — Grokking: Watching a Network Suddenly Understand

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab06_grokking.ipynb)

**⏱ Time:** ~1.5 hours  **🐝 Narrator:** Buzz the Backprop Bee

### 🎯 Learning objectives
- Reproduce **grokking**: a network hits 100% *training* accuracy early, sits at chance-level *test* accuracy for a long time, then abruptly generalizes.
- Show experimentally that **weight decay** is the dial that controls whether (and when) grokking happens.
- Do a **mechanistic peek**: Fourier-analyse the learned embeddings and find the handful of **key frequencies** the network uses to do modular arithmetic (Nanda et al., 2023).
- Track a hidden **progress measure** that moves *before* test accuracy does.
- Fit a **scaling law** $L(N) = aN^{-\alpha} + c$ to your own models, and derive **compute-optimal** model/data sizes from the Chinchilla parametric fit (Hoffmann et al., 2022).

### 🧰 Prerequisites
PyTorch training loops, AdamW & weight decay (Lab 2.2), the discrete Fourier transform at the "it decomposes a signal into sines and cosines" level.

### 🌍 Why this matters in 2026
Grokking is the cleanest laboratory example of a question that matters enormously for frontier models: **when does a network memorize, and when does it find the real algorithm?** It shows that training loss can be flat while the network's *internals* are reorganizing — the founding observation for mechanistic interpretability and for "progress measures" that predict capability jumps. Scaling laws, meanwhile, are how every lab decides how big a model to train and on how many tokens before spending millions of GPU-hours. Today you'll see both: the microscopic (one circuit in one tiny net) and the macroscopic (loss as a smooth function of scale). 🍯

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 | Setup | 3 |
| 1 | Build the modular-addition dataset (🧪 #1) | 7 |
| 2 | Build a tiny embedding + quadratic-activation network (🧪 #2) | 7 |
| 3 | Train far past memorization and watch test accuracy jump | 15 |
| 4 | Ablate weight decay — the dial that controls grokking | 10 |
| 5 | Fourier analysis of the embeddings (🧪 #3), the "clock", and an animation of it forming 🎬 | 20 |
| 6 | Scaling laws: fit $L(N)$ to your own models (🧪 #4) and derive the Chinchilla frontier (🧪 #5) | 25 |

## 0. Setup
Pure PyTorch + numpy + scipy — everything is preinstalled on Colab. The main grokking run takes about 30–60 s on a laptop CPU.

In [ ]:
import os, math, random, time
import numpy as np
import torch
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(0)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
print(f"torch {torch.__version__} | device={DEVICE} | FAST={FAST}")

import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import curve_fit

## Part 1 — The task: clock arithmetic 🕐

We teach a network the function $(a, b) \mapsto (a + b) \bmod p$ for a prime $p$. There are only $p^2$ possible inputs, so we can list *all* of them, train on a random fraction, and test on the rest. There is no noise, no ambiguity — the only way to get the test set right is to learn the actual rule.

Why is this interesting? Because a big enough network can trivially **memorize** the training table — and it does, fast. The question is what happens *after* that.

We'll use $p = 97$ and train on **30%** of the table (FAST mode: $p=43$, 50%).

### 🧪 Your Turn #1 — build the dataset
Create all pairs $(a,b)$, their labels, and a random train/test split of the pair indices.

**Hints:** `torch.meshgrid(torch.arange(p), torch.arange(p), indexing="ij")` then `.flatten()`; `torch.randperm(p*p, generator=g)`; the first `int(frac * p*p)` indices are train.

In [ ]:
# 🧪 YOUR TURN
def make_modular_dataset(p, frac, seed=0):
    g = torch.Generator().manual_seed(seed)
    a = b = y = train_idx = test_idx = None   # TODO
    return a, b, y, train_idx, test_idx

In [ ]:
# ✅ Solution (peek only after trying!)
def make_modular_dataset(p, frac, seed=0):
    g = torch.Generator().manual_seed(seed)
    a, b = torch.meshgrid(torch.arange(p), torch.arange(p), indexing="ij")
    a, b = a.flatten(), b.flatten()
    y = (a + b) % p
    perm = torch.randperm(p * p, generator=g)
    n_train = int(frac * p * p)
    return a, b, y, perm[:n_train], perm[n_train:]

In [ ]:
P, FRAC = (43, 0.5) if FAST else (97, 0.3)
a_all, b_all, y_all, train_idx, test_idx = make_modular_dataset(P, FRAC)

# ✔️ Checkpoint
assert train_idx is not None, "make_modular_dataset still returns None — fill in the TODO"
assert len(train_idx) + len(test_idx) == P * P and len(set(train_idx.tolist()) & set(test_idx.tolist())) == 0
assert (y_all == (a_all + b_all) % P).all()
print(f"p={P}: {len(train_idx)} train pairs, {len(test_idx)} test pairs (chance accuracy = {1/P:.1%})")

table = y_all.view(P, P).numpy()
mask = np.zeros(P * P); mask[train_idx.numpy()] = 1
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(table, cmap="twilight"); axes[0].set_title("(a + b) mod p — a 'diagonal rainbow'")
axes[0].set_xlabel("b"); axes[0].set_ylabel("a")
axes[1].imshow(mask.reshape(P, P), cmap="YlOrBr"); axes[1].set_title(f"which cells are in the training set ({FRAC:.0%})")
axes[1].set_xlabel("b"); plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** With only $p^2$ possible inputs, a network can *memorize* the training table perfectly — so test accuracy on held-out pairs is a clean measure of whether it learned the actual rule.

## Part 2 — The model 🧠

The original grokking paper (Power et al., 2022) and Nanda et al. (2023) used small transformers. Transformers grok too, but they are several times slower per step on a CPU, so here we use the minimal architecture known to grok on modular arithmetic (in the spirit of Gromov, 2023):

$$h = \big(W\,[E_a \,;\, E_b]\big)^{\odot 2},\qquad \text{logits} = U h$$

- $E\in\mathbb R^{p\times d}$: a learned **embedding** for each number (this is what we'll Fourier-analyse later),
- $[E_a; E_b]$: concatenate the two embeddings,
- $(\cdot)^{\odot 2}$: an elementwise **square** activation,
- $U$: unembedding to $p$ logits.

Why squaring? Products are exactly what you need for trigonometry: $\cos(\omega a)\cos(\omega b) - \sin(\omega a)\sin(\omega b) = \cos(\omega(a+b))$. A quadratic of sums contains all those cross terms. (A ReLU MLP can approximate products too — it just takes more neurons and more steps.)

### 🧪 Your Turn #2 — the forward pass
**Hint:** `x = torch.cat([self.E(a), self.E(b)], dim=-1)`, then `self.U(self.W(x) ** 2)`.

In [ ]:
# 🧪 YOUR TURN
class ModAddNet(nn.Module):
    def __init__(self, p, d=64, hidden=256):
        super().__init__()
        self.E = nn.Embedding(p, d)
        self.W = nn.Linear(2 * d, hidden, bias=False)
        self.U = nn.Linear(hidden, p, bias=False)
    def forward(self, a, b):
        pass   # TODO

In [ ]:
# ✅ Solution (peek only after trying!)
class ModAddNet(nn.Module):
    def __init__(self, p, d=64, hidden=256):
        super().__init__()
        self.E = nn.Embedding(p, d)
        self.W = nn.Linear(2 * d, hidden, bias=False)
        self.U = nn.Linear(hidden, p, bias=False)
    def forward(self, a, b):
        x = torch.cat([self.E(a), self.E(b)], dim=-1)
        return self.U(self.W(x) ** 2)

In [ ]:
# ✔️ Checkpoint — shape check, and the output really is a quadratic form of the embeddings
m = ModAddNet(P)
out = m(a_all[:5], b_all[:5])
assert out is not None and out.shape == (5, P)
with torch.no_grad():
    x = torch.cat([m.E(a_all[:5]), m.E(b_all[:5])], -1)
    assert torch.allclose(out, (x @ m.W.weight.T) ** 2 @ m.U.weight.T, atol=1e-5)
print("parameters:", sum(p_.numel() for p_ in m.parameters()))

> **🍯 Key takeaway:** Architecture choice shapes what is *easy* to learn: products of embeddings are exactly what trig identities need, so a squared activation makes the 'clock' algorithm easy to express.

## Part 3 — Train way past the point of "done" ⏳

Recipe (verified to grok on a laptop CPU in ~30–60 s):
- **full-batch** training (the whole training table each step — it's tiny),
- **AdamW**, lr $3\times10^{-3}$, $\beta = (0.9, 0.98)$, **weight decay $\lambda = 1.0$** (that is *huge* by normal standards),
- 3,500 steps — roughly 15× longer than it takes to memorize the training set.

We log train/test accuracy and loss, the total weight norm, and a "Fourier sparsity" score of the embedding (explained in Part 5) every few steps.

### 🔮 Predict-then-run
Training accuracy will hit 100% within ~200 steps. Sketch what you think **test** accuracy does over the remaining ~3,300 steps. Gradual climb? Stuck at chance forever? Something else?

In [ ]:
def fourier_power(E):
    """Fraction of (non-constant) embedding power at each frequency k = 1..(p-1)/2.
    E: (p, d) array. rfft over the token axis; combine all d columns."""
    Fk = np.fft.rfft(E, axis=0)          # (p//2 + 1, d), complex
    power = (np.abs(Fk) ** 2).sum(axis=1)
    power = power[1:]                      # drop k=0 (the mean embedding)
    return power / power.sum()

def topk_fraction(E, k=5):
    return np.sort(fourier_power(E))[::-1][:k].sum()

def train_grokking(p=P, frac=FRAC, wd=1.0, lr=3e-3, steps=None, log_every=10, seed=0, verbose=True, n_snapshots=0):
    steps = steps or (2000 if FAST else 3500)
    a, b, y, tr, te = make_modular_dataset(p, frac, seed=0)        # same split for every run
    a, b, y = a.to(DEVICE), b.to(DEVICE), y.to(DEVICE)
    torch.manual_seed(seed)
    model = ModAddNet(p).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=(0.9, 0.98))
    E0 = model.E.weight.detach().cpu().numpy().copy()
    log = {k: [] for k in ["step", "train_acc", "test_acc", "train_loss", "test_loss", "wnorm", "fourier_top5"]}
    snap_steps = set(np.unique(np.geomspace(1, steps + 1, n_snapshots).astype(int) - 1)) if n_snapshots else set()
    snapshots = []                                    # (step, embedding copy) for the animation in Part 5
    t0 = time.time()
    for step in range(steps + 1):
        logits = model(a[tr], b[tr])
        loss = F.cross_entropy(logits, y[tr])
        if step % log_every == 0:
            with torch.no_grad():
                te_logits = model(a[te], b[te])
                log["step"].append(step)
                log["train_acc"].append((logits.argmax(-1) == y[tr]).float().mean().item())
                log["test_acc"].append((te_logits.argmax(-1) == y[te]).float().mean().item())
                log["train_loss"].append(loss.item())
                log["test_loss"].append(F.cross_entropy(te_logits, y[te]).item())
                log["wnorm"].append(torch.sqrt(sum((w ** 2).sum() for w in model.parameters())).item())
                log["fourier_top5"].append(topk_fraction(model.E.weight.detach().cpu().numpy()))
            if verbose and step % 500 == 0:
                print(f"step {step:5d} | train acc {log['train_acc'][-1]:.3f} | test acc {log['test_acc'][-1]:.3f} "
                      f"| ‖θ‖ {log['wnorm'][-1]:.1f} | {time.time() - t0:.0f}s")
        if step in snap_steps:
            snapshots.append((step, model.E.weight.detach().cpu().numpy().copy()))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    log = {k: np.array(v) for k, v in log.items()}
    model.snapshots = snapshots
    return model, log, E0

model, log, E_init = train_grokking(wd=1.0, n_snapshots=60)

In [ ]:
def first_step(log, key, thresh):
    hit = np.where(log[key] >= thresh)[0]
    return int(log["step"][hit[0]]) if len(hit) else None

mem_step = first_step(log, "train_acc", 0.99)
grok_step = first_step(log, "test_acc", 0.95)
print(f"memorized (train acc ≥ 99%) at step {mem_step};  generalized (test acc ≥ 95%) at step {grok_step}")

s = np.maximum(log["step"], 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(s, log["train_acc"], label="train", color=HIVE[4], lw=2)
axes[0].plot(s, log["test_acc"], label="test", color=HIVE[2], lw=2)
axes[0].set_xscale("log"); axes[0].set_xlabel("step (log scale)"); axes[0].set_ylabel("accuracy")
axes[0].set_title("Grokking: memorize first, understand much later")
if mem_step: axes[0].axvline(mem_step, color=HIVE[4], ls=":", lw=1)
if grok_step: axes[0].axvline(grok_step, color=HIVE[2], ls=":", lw=1)
axes[0].legend()
axes[1].plot(s, log["train_loss"], label="train loss", color=HIVE[4], lw=2)
axes[1].plot(s, log["test_loss"], label="test loss", color=HIVE[2], lw=2)
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("step (log scale)")
axes[1].set_title("Loss: test loss rises while memorizing, then collapses"); axes[1].legend()
plt.tight_layout(); plt.show()

# ✔️ Checkpoint — memorization happens long before generalization, and generalization does happen
assert mem_step is not None and grok_step is not None, "no grokking? check lr / weight decay / steps"
assert grok_step >= 3 * mem_step, "generalization should lag memorization by a wide margin"
assert log["test_acc"][-1] > 0.9
print(f"✔️ Grokked! Generalization arrived {grok_step / mem_step:.0f}× later than memorization. 🐝")

**What just happened?** For a long stretch the training loss is ~0 and test accuracy is at (or even *below*) chance: the network has a lookup table. But weight decay keeps pressing on the weights. A memorizing solution needs many large weights (one "entry" per training example); a *general* algorithm can fit the same training data with much smaller weights. Gradient descent + weight decay slowly trades the former for the latter, and once the generalizing circuit dominates the logits, test accuracy shoots up. (Note the log-x axis: on a linear axis the plateau looks even longer.)

You may also see a few sharp loss spikes *after* grokking. With Adam at a fairly high learning rate and near-zero training loss this is common (it has been studied as the "slingshot" effect); the network recovers within a few dozen steps.

> **🍯 Key takeaway:** Zero training loss is not the end of learning: the network kept reorganizing for thousands of steps after memorizing, and only then generalized.

## Part 4 — Ablation: weight decay is the dial 🎛️

Theory says weight decay supplies the pressure toward the low-norm, generalizing solution. If that's right, then *less* decay ⇒ later (or no) grokking, *more* decay ⇒ earlier grokking.

### 🔮 Predict-then-run
For $\lambda \in \{0,\ 0.3,\ 1,\ 3\}$, which runs will grok within our step budget? Will $\lambda=0$ *ever* generalize here?

In [ ]:
WDS = [0.0, 1.0] if FAST else [0.0, 0.3, 1.0, 3.0]
ablation, ablation_models = {1.0: log}, {1.0: model}
for wd in WDS:
    if wd in ablation:
        continue
    t0 = time.time()
    mdl, lg, _ = train_grokking(wd=wd, verbose=False)
    ablation[wd], ablation_models[wd] = lg, mdl
    print(f"wd={wd:<4} final test acc {lg['test_acc'][-1]:.3f}  grok step {first_step(lg, 'test_acc', 0.95)}  [{time.time() - t0:.0f}s]")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cmap = {0.0: HIVE[1], 0.3: HIVE[4], 1.0: HIVE[2], 3.0: HIVE[0]}
for wd in sorted(ablation):
    lg = ablation[wd]; s = np.maximum(lg["step"], 1)
    axes[0].plot(s, lg["test_acc"], label=f"λ = {wd}", color=cmap[wd], lw=2)
    axes[1].plot(s, lg["wnorm"], label=f"λ = {wd}", color=cmap[wd], lw=2)
axes[0].set_xscale("log"); axes[0].set_title("test accuracy vs step"); axes[0].set_xlabel("step (log)"); axes[0].legend()
axes[1].set_xscale("log"); axes[1].set_title("total weight norm ‖θ‖₂"); axes[1].set_xlabel("step (log)"); axes[1].legend()
plt.tight_layout(); plt.show()

Look at the right panel: without weight decay the weight norm *grows* (Adam keeps pushing logits of a memorized table ever higher); with decay it shrinks, and the grokking moment coincides with a sharp drop in norm — the "cleanup" phase in which the memorization components get pruned away. In this setting stronger decay ⇒ earlier grokking. (Far too much decay will eventually stop the network from fitting even the training set — try a few very large values of $\lambda$ to find where that happens.)

> Weight decay is not the *only* route to grokking — Liu et al. ("Omnigrok", 2022) tie it to the weight norm more broadly (large initialization ⇒ slower grokking), and later work links it to the network moving from a "lazy" to a "rich" feature-learning regime — but it is the most reliable knob.

> **🍯 Key takeaway:** Weight decay supplies the pressure toward the low-norm, general solution: no decay → memorization forever (in our budget); more decay → earlier grokking.

## Part 5 — Mechanistic peek: the network learned Fourier features 🔬

Nanda et al. (2023) reverse-engineered transformers that grokked modular addition and found a beautiful algorithm:

1. **Embed** each number $a$ as waves at a few *key frequencies* $\omega_k = 2\pi k/p$: $\ a \mapsto (\cos\omega_k a, \sin\omega_k a)$ — i.e. a point on a circle, like a clock hand.
2. **Combine** using trig identities to get $\cos(\omega_k(a+b))$ and $\sin(\omega_k(a+b))$ — "add the angles of two clock hands".
3. **Read out** logit$(c) \approx \sum_k \cos\big(\omega_k(a + b - c)\big)$, which is maximal exactly when $c \equiv a+b \pmod p$ (constructive interference across the key frequencies).

**Test it:** if the embedding uses only a few frequencies, the discrete Fourier transform of $E$ along the token axis should be **sparse** — most of the power in a handful of $k$'s. A random (or memorizing) embedding spreads power over all $(p-1)/2$ frequencies.

$$\hat E_k = \sum_{a=0}^{p-1} E_a\, e^{-2\pi i k a / p},\qquad \text{power}(k) = \|\hat E_k\|^2 .$$

### 🧪 Your Turn #3 — Fourier power spectrum
You already used `fourier_power` above (as the progress measure) — now write it yourself.

**Hints:** `np.fft.rfft(E, axis=0)` gives frequencies $k=0..\lfloor p/2\rfloor$ (for real signals, $k$ and $p-k$ carry the same information); sum $|\cdot|^2$ over the embedding dimension; drop $k=0$; normalize to sum to 1.

In [ ]:
# 🧪 YOUR TURN
def my_fourier_power(E):
    power = np.ones(E.shape[0] // 2) / (E.shape[0] // 2)   # TODO: replace this uniform placeholder
    return power

In [ ]:
# ✅ Solution (peek only after trying!)
def my_fourier_power(E):
    Fk = np.fft.rfft(E, axis=0)
    power = (np.abs(Fk) ** 2).sum(axis=1)[1:]
    return power / power.sum()

In [ ]:
# ✔️ Checkpoint
E_final = model.E.weight.detach().cpu().numpy()
assert np.allclose(my_fourier_power(E_final), fourier_power(E_final))
print("✔️ Your Fourier power spectrum matches.")

In [ ]:
E_mem = ablation_models[0.0].E.weight.detach().cpu().numpy()   # the λ=0 memorizer from Part 4

panels = [("random init", E_init), ("after grokking (λ=1)", E_final), ("memorizer (λ=0)", E_mem)]
fig, axes = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 3.8), sharey=True)
ks = np.arange(1, P // 2 + 1)
for ax, (name, E) in zip(axes, panels):
    pw = fourier_power(E)
    ax.bar(ks, pw, color=HIVE[0], edgecolor=HIVE[1], lw=0.3)
    ax.set_title(f"{name}\ntop-5 freqs hold {np.sort(pw)[::-1][:5].sum():.0%} of power")
    ax.set_xlabel("frequency k")
axes[0].set_ylabel("fraction of embedding power")
plt.tight_layout(); plt.show()

key_freqs = np.argsort(fourier_power(E_final))[::-1][:5] + 1
print("key frequencies k:", sorted(key_freqs.tolist()))

# ✔️ Checkpoint — grokked embeddings are Fourier-sparse
top5_grok, top5_init = topk_fraction(E_final), topk_fraction(E_init)
assert top5_grok > 0.5 and top5_grok > 2 * top5_init
print(f"✔️ top-5 frequency share: init {top5_init:.0%} → grokked {top5_grok:.0%}")

Now the "clock": project each number's embedding onto the cosine and sine waves of the strongest key frequency. If the network really represents $a$ as an angle $\omega_k a$, the 97 points should sit on a **circle**, ordered by $k\cdot a \bmod p$.

In [ ]:
k_star = int(key_freqs[0])
tok = np.arange(P)
cos_basis, sin_basis = np.cos(2 * np.pi * k_star * tok / P), np.sin(2 * np.pi * k_star * tok / P)
# the 2-D subspace of embedding space carrying frequency k*
Ec = E_final - E_final.mean(0)
dir_c, dir_s = cos_basis @ Ec, sin_basis @ Ec
basis, _ = np.linalg.qr(np.stack([dir_c, dir_s], 1))    # orthonormal 2-D basis in R^d
xy = Ec @ basis

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (title, pts) in zip(axes, [("random init", (E_init - E_init.mean(0)) @ basis), (f"after grokking — frequency k={k_star}", xy)]):
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=(k_star * tok) % P, cmap="twilight", s=40, edgecolor="k", lw=0.3)
    for t in [0, 1, 2, 3]:
        ax.annotate(str(t), pts[t], fontsize=9, xytext=(4, 4), textcoords="offset points")
    ax.set_aspect("equal"); ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(sc, ax=axes, label=f"({k_star}·a) mod p", shrink=0.8)
plt.show()

### 🎬 Watch the clock assemble itself
We saved 60 snapshots of the embedding during the main run (log-spaced in time). Each frame projects all $p$ embeddings onto the *final* key-frequency plane. Watch the right panel: the circle forms while test accuracy is still flat — then the cliff.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

snaps = model.snapshots
cols = (k_star * tok) % P
final_pts = (E_final - E_final.mean(0)) @ basis
lim = 1.25 * np.abs(final_pts).max()
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6))
pts0 = (snaps[0][1] - snaps[0][1].mean(0)) @ basis
sc = axL.scatter(pts0[:, 0], pts0[:, 1], c=cols, cmap="twilight", s=36, edgecolor="k", lw=0.3)
axL.set_xlim(-lim, lim); axL.set_ylim(-lim, lim); axL.set_aspect("equal"); axL.set_xticks([]); axL.set_yticks([])
s_ = np.maximum(log["step"], 1)
axR.plot(s_, log["train_acc"], color=HIVE[4], lw=2, label="train acc")
axR.plot(s_, log["test_acc"], color=HIVE[2], lw=2, label="test acc")
axR.plot(s_, log["fourier_top5"], color=HIVE[0], lw=2, label="top-5 Fourier share")
axR.set_xscale("log"); axR.set_xlabel("step (log)"); axR.legend(loc="upper left", fontsize=9)
vline = axR.axvline(1, color="k", lw=1.2)
def update(i):
    st, E = snaps[i]
    sc.set_offsets((E - E.mean(0)) @ basis)
    vline.set_xdata([max(st, 1)] * 2)
    axL.set_title(f"embeddings on the k={k_star} plane — step {st}")
    return sc, vline
anim = animation.FuncAnimation(fig, update, frames=len(snaps), interval=110)
plt.close(fig)
display(HTML(anim.to_jshtml(default_mode="once")))

### A progress measure that moves *before* the jump
Remember we logged `fourier_top5` during training. Plot it next to test accuracy: the embedding's Fourier sparsity starts climbing **gradually**, well before the test-accuracy cliff. Nanda et al. made this precise with "restricted" and "excluded" losses, splitting training into three phases: **memorization → circuit formation → cleanup**. Grokking only *looks* sudden from the outside.

In [ ]:
s = np.maximum(log["step"], 1)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(s, log["test_acc"], color=HIVE[2], lw=2, label="test accuracy")
ax.plot(s, log["fourier_top5"], color=HIVE[0], lw=2.5, label="embedding power in top-5 frequencies")
ax.plot(s, log["wnorm"] / log["wnorm"].max(), color=HIVE[1], lw=1.2, ls="--", label="weight norm (normalized)")
ax.set_xscale("log"); ax.set_xlabel("step (log)"); ax.legend(loc="center left")
ax.set_title("Hidden progress: Fourier structure forms before test accuracy jumps")
plt.show()

> **🍯 Key takeaway:** The grokked network implements a readable algorithm — numbers as points on circles at a few key frequencies, addition as rotation — and its formation is visible in the weights long before test accuracy moves.

## Part 6 — Zooming out: scaling laws 📈

Grokking is about *one* model over *time*. Scaling laws are about *many* models over *size*: across many orders of magnitude, test loss is well described by a power law plus a floor,
$$L(N) \approx a\,N^{-\alpha} + c,$$
where $N$ is the parameter count, $\alpha$ the scaling exponent and $c$ the **irreducible loss** (the entropy of the data — no model can beat it). Kaplan et al. (2020) documented this for language models over 7 orders of magnitude.

### 6a. Fit a scaling law to models *you* train
We use a teacher–student setup so the irreducible loss is known exactly: a fixed random "teacher" network generates targets $y = T(x) + \varepsilon$ with noise variance $\sigma^2 = 0.01$, so the best possible MSE is $c^\star = 0.01$. Students of increasing width are trained for a fixed, short budget on fresh samples. (Real scaling studies vary data and compute too — we hold them fixed and vary only $N$.)

In [ ]:
D_IN, SIGMA = 8, 0.1
torch.manual_seed(0)
teacher = nn.Sequential(nn.Linear(D_IN, 128), nn.Tanh(), nn.Linear(128, 128), nn.Tanh(), nn.Linear(128, 1))
with torch.no_grad():
    for mod in teacher:
        if isinstance(mod, nn.Linear):
            nn.init.normal_(mod.weight, std=1.5 / mod.in_features ** 0.5)
teacher.requires_grad_(False)

def sample(n, g):
    x = torch.randn(n, D_IN, generator=g)
    return x, teacher(x) + SIGMA * torch.randn(n, 1, generator=g)

X_eval, y_eval = sample(20000, torch.Generator().manual_seed(1))
WIDTHS = [2, 8, 32, 128] if FAST else [2, 4, 8, 16, 32, 64, 128, 256]
STEPS_SL = 300 if FAST else 1500

scaling = []
t0 = time.time()
for w in WIDTHS:
    torch.manual_seed(0)
    student = nn.Sequential(nn.Linear(D_IN, w), nn.Tanh(), nn.Linear(w, w), nn.Tanh(), nn.Linear(w, 1))
    opt = torch.optim.Adam(student.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS_SL)
    g = torch.Generator().manual_seed(2)
    for _ in range(STEPS_SL):
        xb, yb = sample(256, g)
        loss = F.mse_loss(student(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    with torch.no_grad():
        L = F.mse_loss(student(X_eval), y_eval).item()
    N = sum(p_.numel() for p_ in student.parameters())
    scaling.append((N, L))
    print(f"width {w:4d}  N = {N:7,d}  test MSE = {L:.4f}   [{time.time() - t0:.0f}s]")
N_arr, L_arr = map(np.array, zip(*scaling))

### 🧪 Your Turn #4 — fit the power law
Use `scipy.optimize.curve_fit` to fit $L(N) = aN^{-\alpha} + c$.

**Hints:** define `power_law(N, a, alpha, c)`; good initial guesses help a lot with power laws: `p0=[L_arr[0] * N_arr[0] ** 0.3, 0.3, L_arr.min() / 2]`; add `bounds=([0, 0, 0], [np.inf, 3, L_arr.min()])` so the floor stays below the best observed loss.

In [ ]:
# 🧪 YOUR TURN
def power_law(N, a, alpha, c):
    return a * N ** (-alpha) + c

a_fit, alpha_fit, c_fit = 1.0, 0.0, 0.0   # TODO: replace with curve_fit(...)

In [ ]:
# ✅ Solution (peek only after trying!)
def power_law(N, a, alpha, c):
    return a * N ** (-alpha) + c

(a_fit, alpha_fit, c_fit), _ = curve_fit(power_law, N_arr, L_arr,
                                         p0=[L_arr[0] * N_arr[0] ** 0.3, 0.3, L_arr.min() / 2],
                                         bounds=([0, 0, 0], [np.inf, 3, L_arr.min()]), maxfev=20000)
print(f"fit: L(N) = {a_fit:.3f} · N^(-{alpha_fit:.3f}) + {c_fit:.4f}    (true noise floor σ² = {SIGMA**2:.3f})")

In [ ]:
# ✔️ Checkpoint
pred = power_law(N_arr, a_fit, alpha_fit, c_fit)
r2 = 1 - ((L_arr - pred) ** 2).sum() / ((L_arr - L_arr.mean()) ** 2).sum()
assert r2 > 0.95 and alpha_fit > 0
print(f"✔️ power law explains R² = {r2:.3f} of the variance")

Ns = np.logspace(np.log10(N_arr.min()) - 0.2, np.log10(N_arr.max()) + 1, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].loglog(N_arr, L_arr, "o", color=HIVE[2], ms=8, mec="k", label="trained students")
axes[0].loglog(Ns, power_law(Ns, a_fit, alpha_fit, c_fit), color=HIVE[1], label=f"fit: α = {alpha_fit:.2f}")
axes[0].axhline(SIGMA ** 2, color=HIVE[3], ls="--", label="true irreducible loss σ²")
axes[0].set_xlabel("parameters N"); axes[0].set_ylabel("test MSE"); axes[0].legend(); axes[0].set_title("Loss vs size")
axes[1].loglog(N_arr, L_arr - c_fit, "o", color=HIVE[2], ms=8, mec="k")
axes[1].loglog(Ns, a_fit * Ns ** (-alpha_fit), color=HIVE[1])
axes[1].set_xlabel("parameters N"); axes[1].set_ylabel("L − c (reducible loss)")
axes[1].set_title("Subtract the floor → a straight line on log-log")
plt.tight_layout(); plt.show()

Caveat worth internalizing: our biggest students are **under-trained** for their size (fixed 1,500 steps), so their loss bends toward a step-limited plateau rather than the true floor — the fitted $c$ partly absorbs that. That is precisely the trap Chinchilla exposed in earlier LLM scaling studies: if you don't scale data/steps with model size, you mis-estimate what bigger models can do.

### 6b. Compute-optimal training (Chinchilla) — ⚠️ synthetic data
Hoffmann et al. (2022) modelled loss as a function of **both** parameters $N$ and training tokens $D$:
$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}},$$
and reported (their "Approach 3" fit) $E = 1.69,\ A = 406.4,\ B = 410.7,\ \alpha = 0.34,\ \beta = 0.28$. Training compute is $C \approx 6ND$ FLOPs. For a fixed $C$, a too-big model sees too few tokens and a too-small model wastes data — there's an optimum.

> ⚠️ **Everything below is SYNTHETIC**: curves are generated from the published parametric formula, not from real training runs.

Minimizing $L$ subject to $6ND = C$ gives (Lagrange multipliers — try it!)
$$N_{\text{opt}}(C) = G\left(\frac{C}{6}\right)^{\frac{\beta}{\alpha+\beta}},\quad D_{\text{opt}}(C) = G^{-1}\left(\frac{C}{6}\right)^{\frac{\alpha}{\alpha+\beta}},\quad G = \left(\frac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}}.$$

### 🧪 Your Turn #5 — the compute-optimal frontier
Implement `chinchilla_opt(C)` returning `(N_opt, D_opt)` with the closed form above. The checkpoint compares it with brute-force minimization along an isoFLOP curve.

In [ ]:
E_, A_, B_, ALPHA, BETA = 1.69, 406.4, 410.7, 0.34, 0.28     # Hoffmann et al. 2022, Approach 3
def chin_loss(N, D): return E_ + A_ / N ** ALPHA + B_ / D ** BETA

# 🧪 YOUR TURN
def chinchilla_opt(C):
    N_opt = D_opt = None   # TODO
    return N_opt, D_opt

In [ ]:
# ✅ Solution (peek only after trying!)
def chinchilla_opt(C):
    G = (ALPHA * A_ / (BETA * B_)) ** (1 / (ALPHA + BETA))
    N_opt = G * (C / 6) ** (BETA / (ALPHA + BETA))
    D_opt = (C / 6) ** (ALPHA / (ALPHA + BETA)) / G
    return N_opt, D_opt

In [ ]:
# ✔️ Checkpoint vs brute force along the isoFLOP curve
assert chinchilla_opt(1e20)[0] is not None, "chinchilla_opt still returns None — fill in the TODO"
for C in [1e20, 1e22, 1e24]:
    Ngrid = np.logspace(6, 13, 20001)
    N_bf = Ngrid[np.argmin(chin_loss(Ngrid, C / (6 * Ngrid)))]
    N_cf, D_cf = chinchilla_opt(C)
    assert abs(np.log10(N_bf) - np.log10(N_cf)) < 0.01 and abs(6 * N_cf * D_cf / C - 1) < 1e-9
print("✔️ closed form matches brute force")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
budgets = [1e19, 1e20, 1e21, 1e22, 1e23, 1e24]
Ngrid = np.logspace(7, 12.5, 400)
for i, C in enumerate(budgets):
    col = plt.cm.YlOrBr(0.35 + 0.12 * i)
    axes[0].semilogx(Ngrid, chin_loss(Ngrid, C / (6 * Ngrid)), color=col, lw=2, label=f"C = {C:.0e}")
    No, Do = chinchilla_opt(C)
    axes[0].plot(No, chin_loss(No, Do), "o", color=HIVE[2], mec="k", zorder=5)
axes[0].set_ylim(1.8, 4.0); axes[0].set_xlabel("parameters N"); axes[0].set_ylabel("predicted loss")
axes[0].set_title("SYNTHETIC isoFLOP curves (Chinchilla fit)"); axes[0].legend(fontsize=8)

Cs = np.logspace(18, 26, 50)
No, Do = chinchilla_opt(Cs)
axes[1].loglog(Cs, No, color=HIVE[4], lw=2.5, label=f"N_opt ∝ C^{BETA / (ALPHA + BETA):.2f}")
axes[1].loglog(Cs, Do, color=HIVE[2], lw=2.5, label=f"D_opt ∝ C^{ALPHA / (ALPHA + BETA):.2f}")
axes[1].set_xlabel("training compute C (FLOPs)"); axes[1].set_title("SYNTHETIC compute-optimal frontier"); axes[1].legend()
plt.tight_layout(); plt.show()

for C in [5.76e23]:   # roughly Chinchilla's/Gopher's training budget
    n, d = chinchilla_opt(C)
    print(f"C = {C:.2e}: N_opt ≈ {n / 1e9:.0f}B params, D_opt ≈ {d / 1e12:.2f}T tokens, D/N ≈ {d / n:.0f} tokens/param")

**Reading the result.** Notice the Approach-3 fit implies a fairly large tokens-per-parameter ratio at this budget. Its exponents say $N$ and $D$ should grow at *roughly* the same rate with compute (≈0.45 vs ≈0.55); the paper's other two approaches gave ≈0.5/0.5, which is where the famous rule of thumb **~20 training tokens per parameter** comes from (Chinchilla: 70B params, 1.4T tokens). A later replication (Besiroglu et al., 2024) re-fit the parametric model and found estimates more consistent with that ~20:1 ratio — a good reminder that fitted constants are fragile even when the qualitative picture is solid.

**2026 perspective.** "Compute-optimal" minimizes *training* compute for a target loss. But a deployed model is run billions of times, so modern open models (e.g. Llama 3's 8B model trained on ~15T tokens — almost 2,000 tokens/param) are deliberately **over-trained** far past the Chinchilla point: you pay more at training time to get a smaller, cheaper-to-serve model at the same quality. Scaling laws remain the tool for making that trade-off quantitatively — just with inference cost added to the objective.

> **🍯 Key takeaway:** Loss falls as a smooth power law in scale down to an irreducible floor, and for a fixed compute budget there is an optimal split between parameters and data — though deployment costs now push practitioners to over-train smaller models.

## 🏆 Boss Level
1. **Transformer grokking (⚡ GPU recommended).** Replace `ModAddNet` with a 1-layer transformer (inputs `[a, b, =]`, $d=128$, 4 heads, ReLU MLP of width 512, no LayerNorm) following Nanda et al.: $p=113$, 30% train, AdamW lr 1e-3, wd 1.0, $\beta=(0.9, 0.98)$, full batch, ~10–20k steps. Does the embedding become Fourier-sparse in the same way? Which key frequencies does *your* run pick?
2. **Which operations grok?** Try $a-b$, $a\cdot b$ (hint: think about discrete logarithms), and $a^2 + b^2 \bmod p$. Plot grokking time vs. training fraction for each — Power et al. found some operations need much more data.
3. **Data-limited scaling law.** Repeat Part 6a but give each student a *fixed finite dataset* of $D$ examples and vary both $N$ and $D$; fit $L(N, D) = E + A/N^\alpha + B/D^\beta$ to your own runs and find your own compute-optimal frontier.

## 🍯 Key Takeaways
- **Grokking:** train accuracy hits 100% early; test accuracy jumps *much* later. The plateau is not idle — internal structure is forming the whole time.
- **Weight decay** is the main dial: it favours the low-norm *general* solution over the high-norm *memorizing* one.
- **Mechanism:** the embeddings become sparse in a few key frequencies; numbers sit on circles; addition becomes rotation — readable straight off the weights.
- **Progress measures** (like Fourier sparsity or weight norm) can reveal a coming capability jump before the headline metric moves.
- **Scaling laws:** test loss ≈ $aN^{-\alpha} + c$; subtract the floor and you get a straight line on log-log.
- **Chinchilla:** for fixed compute $C \approx 6ND$, grow parameters and tokens together (~20 tokens/param); production models deliberately over-train to cut inference cost.

## 🎤 Explain it to a friend
A student crams a multiplication table by rote, scoring 100% on homework but failing any new question — then weeks later, still studying the same table, suddenly "gets" how multiplication works and aces everything. Explain to a friend how that story maps onto what you saw today (memorization, a long plateau, then generalization), what role "keeping your notes short" (weight decay) plays, and how we could tell from the *inside* of the network that understanding was forming before the test score moved.

## 💼 Interview corner
**1. What is grokking, and what typically makes it happen?**
<details><summary>Answer</summary>

Grokking is delayed generalization: a model fits the training set early (train accuracy ~100%, test at chance) and only much later, with continued training, achieves high test accuracy. It's most reliably observed on small algorithmic datasets with limited training fractions and with regularization — especially weight decay — which gradually favours a low-norm generalizing circuit over a high-norm memorizing one. Nanda et al. split it into memorization, circuit formation and cleanup phases.
</details>

**2. You fit $L(N) = aN^{-\alpha} + c$ to five model sizes. What can go wrong when you extrapolate it?**
<details><summary>Answer</summary>

The fit can be ill-conditioned: $a$, $\alpha$ and $c$ trade off against each other, so small noise changes the extrapolation a lot. If larger models were under-trained (data or steps not scaled with $N$), the curve bends early and you underestimate what bigger models achieve — the issue Chinchilla highlighted in earlier LLM scaling studies. Hyperparameters (LR, batch size) must also be tuned per scale, and a power law fitted over a narrow range may not hold far outside it.
</details>

**3. What did Chinchilla change about how LLMs are trained, and why do many 2024–2026 models ignore its recommendation?**
<details><summary>Answer</summary>

Hoffmann et al. showed that for a fixed training compute budget, parameters and training tokens should scale roughly equally (~20 tokens per parameter), so many earlier models (e.g. Gopher) were too big and under-trained; the 70B Chinchilla trained on 1.4T tokens beat the 280B Gopher with the same compute. But compute-optimal minimizes *training* cost only; when a model will serve huge inference volume, it's cheaper overall to train a smaller model on far more tokens (e.g. Llama 3 8B on ~15T tokens).
</details>

## 🐝 Fun fact
The word *grok* comes from Robert A. Heinlein's 1961 science-fiction novel *Stranger in a Strange Land*, where it is Martian for understanding something so thoroughly that it becomes part of you — a fitting name for a network that suddenly *gets* modular arithmetic.

## 📚 Further reading
- A. Power, Y. Burda, H. Edwards, I. Babuschkin, V. Misra, "Grokking: Generalization Beyond Overfitting on Small Algorithmic Datasets", arXiv:2201.02177 (2022).
- N. Nanda, L. Chan, T. Lieberum, J. Smith, J. Steinhardt, "Progress measures for grokking via mechanistic interpretability", ICLR 2023.
- Z. Liu, E. J. Michaud, M. Tegmark, "Omnigrok: Grokking Beyond Algorithmic Data", ICLR 2023.
- A. Gromov, "Grokking modular arithmetic", arXiv:2301.02679 (2023).
- Z. Zhong, Z. Liu, M. Tegmark, J. Andreas, "The Clock and the Pizza: Two Stories in Mechanistic Explanation of Neural Networks", NeurIPS 2023.
- J. Kaplan et al., "Scaling Laws for Neural Language Models", arXiv:2001.08361 (2020).
- J. Hoffmann et al., "Training Compute-Optimal Large Language Models", NeurIPS 2022.
- T. Besiroglu, E. Erdil, M. Barnett, J. You, "Chinchilla Scaling: A replication attempt", arXiv:2404.10102 (2024).
- T. Kumar, B. Bordelon, S. J. Gershman, C. Pehlevan, "Grokking as the transition from lazy to rich training dynamics", ICLR 2024.